# Model Training

## Objective

The objective of this notebook is to train and compare multiple machine learning classification models for predicting student placement outcomes.

The models used are:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. K-Nearest Neighbors (KNN)
5. Naive Bayes

The models will be compared using Accuracy, Precision, Recall, and F1 Score.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

pd.set_option("display.max_columns", None)

In [2]:
feature_df = pd.read_csv(
    r"C:\Users\rohit\Documents\CareerInsight AI Student Placement Prediction & Analytics Platform\data\processed\student_placement_feature_engineered.csv"
)

print("Dataset loaded successfully!")
print("Shape:", feature_df.shape)

Dataset loaded successfully!
Shape: (12000, 20)


In [3]:
feature_df.head()

,student_id,gender,age,degree,branch,cgpa,backlogs,internships,certifications,coding_skills,communication_skills,aptitude_score,projects,placed,company_type,package_lpa,overall_skill_score,academic_performance_index,profile_strength,profile_development_score
0,1,Male,20,BE,Mechanical,8.40,2,2,2,1,3,69,0,0,NaN,0.0,24.333333,74.0,4,4
1,2,Female,20,BTech,Electrical,8.60,2,0,5,1,9,81,4,0,NaN,0.0,30.333333,76.0,9,9
2,3,Male,22,BCA,Electrical,6.62,3,0,1,1,7,50,1,0,NaN,0.0,19.333333,51.2,2,2
3,4,Male,24,BCA,DS,8.01,0,0,4,4,7,47,4,0,NaN,0.0,19.333333,80.1,8,8
4,5,Male,24,BCA,Electrical,9.12,2,1,2,4,4,77,4,0,NaN,0.0,28.333333,81.2,7,7


In [4]:
print(feature_df.columns.tolist())

['student_id', 'gender', 'age', 'degree', 'branch', 'cgpa', 'backlogs', 'internships', 'certifications', 'coding_skills', 'communication_skills', 'aptitude_score', 'projects', 'placed', 'company_type', 'package_lpa', 'overall_skill_score', 'academic_performance_index', 'profile_strength', 'profile_development_score']


In [19]:
X = feature_df.drop(
    columns=[
        "student_id",
        "placed",
        "company_type",
        "package_lpa"
    ]
)

y = feature_df["placed"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print("placed")

print("\nFeature Shape:", X.shape)
print("Target Shape:", y.shape)

Features:
['gender', 'age', 'degree', 'branch', 'cgpa', 'backlogs', 'internships', 'certifications', 'coding_skills', 'communication_skills', 'aptitude_score', 'projects', 'overall_skill_score', 'academic_performance_index', 'profile_strength', 'profile_development_score']

Target:
placed

Feature Shape: (12000, 16)
Target Shape: (12000,)


In [20]:
print(X.columns.tolist())

['gender', 'age', 'degree', 'branch', 'cgpa', 'backlogs', 'internships', 'certifications', 'coding_skills', 'communication_skills', 'aptitude_score', 'projects', 'overall_skill_score', 'academic_performance_index', 'profile_strength', 'profile_development_score']


In [21]:
print(y.value_counts())

placed
0    8348
1    3652
Name: count, dtype: int64


In [7]:
print(y.value_counts(normalize=True) * 100)

placed
0    69.566667
1    30.433333
Name: proportion, dtype: float64


In [8]:
if y.dtype == "object":
    y = y.astype(str).str.strip().str.lower()

    y = y.map({
        "yes": 1,
        "no": 0,
        "placed": 1,
        "not placed": 0
    })

if y.isnull().any():
    print("Warning: Some target values could not be converted.")
    print(y[y.isnull()].head())

else:
    y = y.astype(int)

print(y.value_counts())

placed
0    8348
1    3652
Name: count, dtype: int64


In [9]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['gender', 'degree', 'branch', 'company_type']

Numerical Features:
['age', 'cgpa', 'backlogs', 'internships', 'certifications', 'coding_skills', 'communication_skills', 'aptitude_score', 'projects', 'overall_skill_score', 'academic_performance_index', 'profile_strength', 'profile_development_score']


In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [12]:
print("Training Data:", X_train.shape)
print("Testing Data :", X_test.shape)

Training Data: (9600, 17)
Testing Data : (2400, 17)


In [13]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Naive Bayes": GaussianNB()
}

In [14]:
pipelines = {}

for name, model in models.items():

    pipelines[name] = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

print("Model pipelines created successfully!")

Model pipelines created successfully!


In [15]:
results = []

trained_models = {}

for name, pipeline in pipelines.items():

    print(f"Training {name}...")

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    trained_models[name] = pipeline

print("\nAll models trained successfully!")

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training KNN...
Training Naive Bayes...

All models trained successfully!


In [16]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,1.000000,1.000000,1.000000,1.000000
1,Decision Tree,1.000000,1.000000,1.000000,1.000000
2,Random Forest,1.000000,1.000000,1.000000,1.000000
3,Naive Bayes,1.000000,1.000000,1.000000,1.000000
4,KNN,0.940417,0.917496,0.883562,0.900209


In [17]:
results_percentage = results_df.copy()

for column in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
]:
    results_percentage[column] = (
        results_percentage[column] * 100
    ).round(2)

results_percentage

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,100.00,100.00,100.00,100.00
1,Decision Tree,100.00,100.00,100.00,100.00
2,Random Forest,100.00,100.00,100.00,100.00
3,Naive Bayes,100.00,100.00,100.00,100.00
4,KNN,94.04,91.75,88.36,90.02


In [18]:
pd.crosstab(
    feature_df["company_type"],
    feature_df["placed"],
    normalize="index"
) * 100

placed,1
company_type,
Product,100.0
Service,100.0
Startup,100.0


In [22]:
feature_df.groupby("placed").mean(numeric_only=True).T

placed,0,1
student_id,5994.565405,6014.065717
age,21.994609,21.973987
cgpa,7.427245,8.143771
backlogs,1.492094,1.490416
internships,1.488860,1.529299
certifications,2.501318,2.526561
coding_skills,4.611164,7.516429
communication_skills,5.511140,5.476451
aptitude_score,64.854816,79.552574
projects,2.530067,2.446878


In [23]:
feature_df["placed"].value_counts(normalize=True) * 100

placed
0    69.566667
1    30.433333
Name: proportion, dtype: float64

In [24]:
print("Placed students:")
print(feature_df[feature_df["placed"] == 1][
    [
        "cgpa",
        "coding_skills",
        "aptitude_score",
        "internships",
        "certifications",
        "projects",
        "backlogs"
    ]
].describe())

print("\nNot Placed students:")
print(feature_df[feature_df["placed"] == 0][
    [
        "cgpa",
        "coding_skills",
        "aptitude_score",
        "internships",
        "certifications",
        "projects",
        "backlogs"
    ]
].describe())

Placed students:
              cgpa  coding_skills  aptitude_score  internships  \
count  3652.000000    3652.000000     3652.000000  3652.000000   
mean      8.143771       7.516429       79.552574     1.529299   
std       0.964798       1.730866       11.447508     1.131200   
min       6.500000       5.000000       60.000000     0.000000   
25%       7.300000       6.000000       70.000000     1.000000   
50%       8.130000       8.000000       80.000000     2.000000   
75%       8.990000       9.000000       90.000000     3.000000   
max       9.800000      10.000000       99.000000     3.000000   

       certifications     projects     backlogs  
count     3652.000000  3652.000000  3652.000000  
mean         2.526561     2.446878     1.490416  
std          1.702981     1.703324     1.120593  
min          0.000000     0.000000     0.000000  
25%          1.000000     1.000000     0.000000  
50%          3.000000     2.000000     1.000000  
75%          4.000000     4.000000    

In [25]:
print("Gender vs Placement")
print(pd.crosstab(
    feature_df["gender"],
    feature_df["placed"],
    normalize="index"
) * 100)

print("\nDegree vs Placement")
print(pd.crosstab(
    feature_df["degree"],
    feature_df["placed"],
    normalize="index"
) * 100)

print("\nBranch vs Placement")
print(pd.crosstab(
    feature_df["branch"],
    feature_df["placed"],
    normalize="index"
) * 100)

Gender vs Placement
placed          0          1
gender                      
Female  69.342457  30.657543
Male    69.790280  30.209720

Degree vs Placement
placed          0          1
degree                      
BCA     71.409964  28.590036
BE      68.299228  31.700772
BSc     68.223043  31.776957
BTech   70.279367  29.720633

Branch vs Placement
placed              0          1
branch                          
AI          69.742063  30.257937
CS          70.205128  29.794872
DS          67.855349  32.144651
Electrical  70.015144  29.984856
IT          70.741688  29.258312
Mechanical  68.913147  31.086853


In [26]:
from sklearn.tree import DecisionTreeClassifier

test_features = [
    "cgpa",
    "coding_skills",
    "aptitude_score"
]

X_test_rule = feature_df[test_features]
y_test_rule = feature_df["placed"]

rule_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

rule_model.fit(X_test_rule, y_test_rule)

rule_predictions = rule_model.predict(X_test_rule)

print(
    "Training Accuracy:",
    accuracy_score(y_test_rule, rule_predictions)
)

Training Accuracy: 1.0


In [27]:
from sklearn.tree import export_text

print(
    export_text(
        rule_model,
        feature_names=[
            "cgpa",
            "coding_skills",
            "aptitude_score"
        ]
    )
)

|--- coding_skills <= 4.50
|   |--- class: 0
|--- coding_skills >  4.50
|   |--- aptitude_score <= 59.50
|   |   |--- class: 0
|   |--- aptitude_score >  59.50
|   |   |--- cgpa <= 6.49
|   |   |   |--- class: 0
|   |   |--- cgpa >  6.49
|   |   |   |--- class: 1



## Deployment model: calibrated probabilities

The comparison above can retain the decision tree for learning purposes, but it must not be deployed: its terminal leaves create overconfident 0%/100% probabilities. The command below trains the deployment model with pre-placement inputs only, excludes outcome-known fields such as `company_type` and `package_lpa`, and saves a calibrated random forest.

In [ ]:
%run ../models/train_placement_model.py
